# レッスン05 - エージェント型RAG


## セットアップ

このノートブックでは、Microsoft Agent Framework を使用した Agentic RAG（Retrieval-Augmented Generation）パターンを示します。

**前提条件:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — Azure AI Search サービスのエンドポイント
- `AZURE_SEARCH_API_KEY` — Azure AI Search API キー
- 環境変数経由で構成された Azure OpenAI デプロイメント
- Azure CLI による認証済み（`az login`）


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAGとは？

従来のRAGは、文書を取得し、その後に応答を生成するという固定されたパイプラインに従います。**Agentic RAG**はさらに進んで、エージェントに情報を取得する**タイミング**や**方法**を自主的に決定する能力を与えます。

Agentic RAGでは、エージェントは以下のことができます：
- 質問に答える前に情報取得が必要かどうかを**判断**
- どのデータソースやツールを問い合わせるかを**選択**
- 取得した結果を**評価**し、最初の試みで不十分な場合は追加の取得を実行
- 複数の取得ステップから得た情報を**統合**して一貫した回答を作成

これにより、固定された取得後生成のパイプラインと比べて、エージェントはより柔軟かつ正確になります。


## 検索ツールの作成

Agentic RAGでは、外部データソースをエージェントが必要に応じて呼び出せる**ツール**としてラップします。これにより、エージェントは取得を必須のステップではなく、実行できるアクションの一つとして扱うことができます。

以下では、旅行に関するナレッジベースを定義し、エージェントが目的地情報を調べるために呼び出せるツールとして公開します。


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAGエージェントの構築

ここで、**必ず回答する前に情報を取得する**よう指示されたエージェントを作成します。エージェントは `search_travel_knowledge` ツールを使用して、自身のトレーニングデータに頼るのではなく、ナレッジベースに基づいて回答を構築します。


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## 反復取得 — メーカー・チェッカーパターン

Agentic RAG の主な利点の一つは **反復取得** です。エージェントは複数回の検索を行い、初期の発見を検証、洗練、または拡張することができます — これは「メーカー・チェッカー」ワークフローに似ています：

1. **メーカーのステップ**：エージェントが初期情報を取得し、回答案を作成します。
2. **チェッカーのステップ**：エージェントが詳細を検証したり欠けている部分を補ったりするために追加の検索を行います。

以下の例では、複数の目的地を比較する必要がある質問がエージェントに与えられ、複数回検索を行うことを促しています。


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

このレッスンでは、Microsoft Agent Framework を使用して **Agentic RAG** システムを構築する方法を学びました。

- **Agentic RAG** は、エージェントが情報を検索するタイミングを自律的に決定できるようにし、検索を固定的ではなく動的にします。
- **ツールとしてのデータソース**：Azure AI Search のような外部ナレッジベースをエージェントが呼び出せるツールとしてラップします。
- **反復的な検索**：メーカー-チェッカー パターンにより、エージェントは最終回答を出す前に複数回の検索と検証、精緻化を行えます。

本番環境では、大規模な旅行関連ドキュメントの検索を扱うために、インメモリの `TRAVEL_KNOWLEDGE_BASE` を実際の Azure AI Search インデックスに置き換えます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を利用して翻訳されました。正確性には努めておりますが、自動翻訳には誤りや不正確な部分が含まれる可能性があります。原文の母語版を権威ある情報源としてご参照ください。重要な情報については、専門の翻訳者による翻訳を推奨いたします。本翻訳の利用により生じたいかなる誤解や誤訳についても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
